In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [2]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [3]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [6]:
import json
import sys
import os

# Add the directory containing the file to the system path
sys.path.append(os.path.abspath("../"))
from evaluation_utils import llm_structured

In [7]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [8]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["filename"]
        })

    return results, usage

In [9]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:3]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage.input_tokens)

  0%|          | 0/3 [00:00<?, ?it/s]

In [10]:
ground_truth, usages

([{'question': 'What is retrieval-augmented generation, and how does it help when an LLM doesn’t know the answer on its own?',
   'document': '01-agentic-rag/lessons/01-intro.md'},
  {'question': 'Why does this course treat large language models like a black box instead of explaining how they work inside?',
   'document': '01-agentic-rag/lessons/01-intro.md'},
  {'question': 'What are the main limits of LLMs that make RAG useful in the first place?',
   'document': '01-agentic-rag/lessons/01-intro.md'},
  {'question': 'What is the FAQ agent in this module supposed to do, and what kind of data will it answer from?',
   'document': '01-agentic-rag/lessons/01-intro.md'},
  {'question': 'What will be covered in the first part of this module versus the second part?',
   'document': '01-agentic-rag/lessons/01-intro.md'},
  {'question': 'What do I need installed before starting this module, and do I need anything besides Python and Jupyter?',
   'document': '01-agentic-rag/lessons/02-environm

**Q1: Answer**

In [11]:
average = sum(usages)/len(usages)
average

1353.0

In [12]:
import pandas as pd

ground_truth = []
ground_truth = pd.read_csv("ground-truth.csv")

In [14]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [ ]:
ground_truth = ground_truth.to_dict(orient='records')
q = ground_truth[0]["question"]

In [35]:
from minsearch import Index

index = Index(
    text_fields = ['content'],
    keyword_fields=["filename"]
)

index.fit(chunks)

In [36]:
def text_search(query, num_results=5):
   
    return index.search(
        query=query,
        filter_dict={},
        boost_dict={'question': 3.0, 'content': 1.0},
        num_results=num_results
    )

**Q2: Answer**

In [ ]:
result = text_search(q)
result

{'start': 3000,
 'content': 'we drop it.\n\nBuild a prompt that includes both the question and the context:\n\n```python\nprompt = f"""\nYour task is to answer questions from the course participants\nbased on the provided context.\n\nUse the context to find relevant information and provide accurate\nanswers. If the answer is not found in the context,\nrespond with "I don\'t know."\n\nQuestion:\n{question}\n\nContext:\n{context}\n"""\n```\n\nInstead of sending the raw question to the LLM, we send this prompt:\n\n```python\nanswer = llm(prompt)\nprint(answer)\n```\n\nAfter that, the answer is correct: "Yes, you can still join. If you want to\nreceive a certificate, you need to submit your project while\nsubmissions are still open."\n\nThis is the answer we actually want to give to our students. What we\njust did is nothing but RAG.\n\n## Retrieval plus generation\n\nRAG stands for Retrieval-Augmented Generation. Generation is the LLM\nproducing text, and retrieval is search. We retrieve 

In [38]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [64]:
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(chunks), batch_size)):
    batch =  chunks[i:i + batch_size]
    batch_vectors = model.encode([batch_item['content'] for batch_item in batch])
    vectors.extend(batch_vectors)

len(vectors)


  0%|          | 0/6 [00:00<?, ?it/s]

295

In [65]:
import numpy as np
X = np.array(vectors)

In [67]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["filename"])
vindex.fit(X, chunks)

In [109]:

def vector_search(query, num_results=5):

    query_vector = model.encode(query)
    results = vindex.search(query_vector, num_results=num_results)

    return results



**Q3: Answer**

In [70]:

q = ground_truth[0]["question"]
results = vector_search(q)
print(results[0]['filename'])

01-agentic-rag/lessons/01-intro.md


In [73]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [90]:
def compute_relevance(q, search_function):
    doc_id = q["filename"]
    results = search_function(q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == doc_id))

    return relevance

In [91]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [92]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

In [93]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

**Q4: Answer**

In [95]:
evaluate(
    ground_truth,
    text_search
)

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.7583333333333333, 'mrr': 0.5942592592592594}

**Q5: Answer**

In [96]:
evaluate(ground_truth, vector_search)

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.8083333333333333, 'mrr': 0.6356944444444446}

In [97]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [ ]:
def hybrid_search(query, k):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

**Q6: Answer**

In [ ]:

k_values = [1, 50, 100, 200]
results = {}

for k in k_values:
    metrics = evaluate(ground_truth, lambda q: hybrid_search(q, k))
    results[k] = metrics['mrr']
    print(f"k={k}, MRR={metrics['mrr']}")

# the best k
best_k = max(results, key=results.get)
print(f"The best k is: {best_k}")

  0%|          | 0/360 [00:00<?, ?it/s]

k=1, MRR=0.6722685185185188


  0%|          | 0/360 [00:00<?, ?it/s]

k=50, MRR=0.6721296296296295


  0%|          | 0/360 [00:00<?, ?it/s]

k=100, MRR=0.6721296296296295


  0%|          | 0/360 [00:00<?, ?it/s]

k=200, MRR=0.6721296296296295
The best k is: 1
